# Data Generation

This notebook generates synthetic ad tech sample data and saves it as CSV files to a Databricks volume for ingestion.

## Setup catalog, schema, and volume

The notebook will create the required Databricks catalog, schema, and volume before generating the demo data.

In [ ]:
# Run Databricks setup statements and create tables before generating data.

spark.sql("CREATE CATALOG IF NOT EXISTS adtech_demo")
spark.sql("CREATE SCHEMA IF NOT EXISTS adtech_demo.adtech_platform")
spark.sql("CREATE VOLUME IF NOT EXISTS adtech_demo.adtech_platform.generated_datasets")

# Create tables if they do not exist
spark.sql("""
CREATE TABLE IF NOT EXISTS adtech_demo.adtech_platform.campaigns (
  campaign_id STRING,
  campaign_name STRING,
  advertiser STRING,
  start_date DATE,
  end_date DATE,
  budget DOUBLE,
  target_audience STRING,
  created_at TIMESTAMP
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS adtech_demo.adtech_platform.impressions (
  impression_id STRING,
  campaign_id STRING,
  publisher STRING,
  placement STRING,
  event_timestamp TIMESTAMP,
  viewability BOOLEAN,
  view_duration_seconds DOUBLE,
  audience_segment STRING
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS adtech_demo.adtech_platform.clicks (
  click_id STRING,
  impression_id STRING,
  campaign_id STRING,
  click_timestamp TIMESTAMP,
  cost DOUBLE,
  device STRING,
  geo STRING
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS adtech_demo.adtech_platform.conversions (
  conversion_id STRING,
  click_id STRING,
  campaign_id STRING,
  conversion_timestamp TIMESTAMP,
  revenue DOUBLE,
  conversion_type STRING
)
USING DELTA
""")

spark.sql("""
CREATE TABLE IF NOT EXISTS adtech_demo.adtech_platform.costs (
  campaign_id STRING,
  cost_date DATE,
  daily_cost DOUBLE,
  channel STRING
)
USING DELTA
""")

import uuid
from datetime import datetime, timedelta
import random
import pandas as pd
import os

# Define volume path (adjust catalog/schema/volume as needed)
volume_path = '/dbfs/Volumes/adtech_demo/adtech_platform/generated_datasets/'
os.makedirs(volume_path, exist_ok=True)

# Generate campaigns
campaigns = [
    {
        'campaign_id': f'cmp_{i:03d}',
        'campaign_name': f'Holiday Launch {i}',
        'advertiser': random.choice(['MediaCo', 'StreamMax', 'SportPlay', 'EntertainmentX']),
        'start_date': (datetime(2025, 11, 1) + timedelta(days=7 * i)).date(),
        'end_date': datetime(2025, 12, 31).date(),
        'budget': round(random.uniform(10000, 250000), 2),
        'target_audience': random.choice(['GenZ', 'Millennials', 'Families', 'Gamers']),
        'created_at': datetime.now()
    }
    for i in range(1, 6)
]

# Generate impressions, clicks, conversions, costs
impressions = []
clicks = []
conversions = []
costs = []

for campaign in campaigns:
    for j in range(200):
        imp_id = str(uuid.uuid4())
        event_ts = datetime(2025, 11, 15) + timedelta(minutes=random.randint(0, 86400))
        view_duration = max(0.0, random.gauss(12, 8))
        viewability = view_duration >= 1.0
        audience = random.choice(['Premium', 'Casual', 'Active', 'Core'])
        impressions.append({
            'impression_id': imp_id,
            'campaign_id': campaign['campaign_id'],
            'publisher': random.choice(['video_site', 'mobile_app', 'streaming_service']),
            'placement': random.choice(['pre-roll', 'mid-roll', 'banner', 'native']),
            'event_timestamp': event_ts,
            'viewability': viewability,
            'view_duration_seconds': round(view_duration, 2),
            'audience_segment': audience
        })
        if random.random() < 0.15:
            click_id = str(uuid.uuid4())
            click_ts = event_ts + timedelta(seconds=random.randint(5, 300))
            cost = round(random.uniform(0.25, 5.0), 2)
            clicks.append({
                'click_id': click_id,
                'impression_id': imp_id,
                'campaign_id': campaign['campaign_id'],
                'click_timestamp': click_ts,
                'cost': cost,
                'device': random.choice(['mobile', 'desktop', 'tablet']),
                'geo': random.choice(['US', 'UK', 'CA', 'AU'])
            })
            if random.random() < 0.35:
                conversions.append({
                    'conversion_id': str(uuid.uuid4()),
                    'click_id': click_id,
                    'campaign_id': campaign['campaign_id'],
                    'conversion_timestamp': click_ts + timedelta(minutes=random.randint(2, 90)),
                    'revenue': round(cost * random.uniform(1.5, 4.0), 2),
                    'conversion_type': random.choice(['video_watch_complete', 'signup', 'in_app_purchase'])
                })
    costs.append({
        'campaign_id': campaign['campaign_id'],
        'cost_date': campaign['start_date'],
        'daily_cost': round(campaign['budget'] / 15, 2),
        'channel': random.choice(['display', 'video', 'social', 'search'])
    })

# Save to CSVs in volume
pd.DataFrame(campaigns).to_csv(volume_path + 'campaigns.csv', index=False)
pd.DataFrame(impressions).to_csv(volume_path + 'impressions.csv', index=False)
pd.DataFrame(clicks).to_csv(volume_path + 'clicks.csv', index=False)
pd.DataFrame(conversions).to_csv(volume_path + 'conversions.csv', index=False)
pd.DataFrame(costs).to_csv(volume_path + 'costs.csv', index=False)

# Load generated data into Delta tables
spark.createDataFrame(pd.DataFrame(campaigns)).write.format('delta').mode('overwrite').saveAsTable('adtech_demo.adtech_platform.campaigns')
spark.createDataFrame(pd.DataFrame(impressions)).write.format('delta').mode('overwrite').saveAsTable('adtech_demo.adtech_platform.impressions')
spark.createDataFrame(pd.DataFrame(clicks)).write.format('delta').mode('overwrite').saveAsTable('adtech_demo.adtech_platform.clicks')
spark.createDataFrame(pd.DataFrame(conversions)).write.format('delta').mode('overwrite').saveAsTable('adtech_demo.adtech_platform.conversions')
spark.createDataFrame(pd.DataFrame(costs)).write.format('delta').mode('overwrite').saveAsTable('adtech_demo.adtech_platform.costs')

print('Sample data generated, saved to volume, and loaded into tables:', volume_path)
print('Files:', os.listdir(volume_path))